# 📗 지식 그래프: 온톨로지·RDF 트리플·SPARQL

앞서 데이터를 노드와 관계로 **모델링**했습니다. 이번 시간에는 그 그래프에 **의미**를 입힙니다. "세종의 국적은 조선", "아인슈타인의 직업은 물리학자" 같은 **사실**을 모아 놓은 그래프를 **지식 그래프**라 부르고, 어떤 노드와 관계를 담을지 정한 규격을 **온톨로지**라 부릅니다.

오늘 쓰는 데이터는 파일이 아닙니다. 세계에서 가장 큰 공개 지식 그래프인 **Wikidata** 에 표준 쿼리 언어 **SPARQL** 로 직접 질의해 받아 옵니다. 사실을 담는 표준 형식 **RDF 트리플(주어-술어-목적어)** 을 배우고, 그 위에서 SPARQL 을 **직접 작성해 실행**합니다.

## ⏪ 복습: 지난 시간

- **LPG 모델**: 노드·레이블·관계(방향)·속성으로 그래프를 dict/list 로 만들었습니다.
- **그래프 순회**: 이웃을 따라가는 1홉·2홉을 반복문으로 구현했습니다.
- 오늘은 그래프에 **규격(온톨로지)** 과 **표준 형식(RDF)**, 그리고 **쿼리 언어(SPARQL)** 를 더합니다.

**오늘의 목표**

- [ ] **지식 그래프**가 무엇인지, 왜 "사실 = 연결"인지 설명한다.
- [ ] 공개 지식 그래프 **Wikidata** 에 질의해 실제 사실을 받아 온다.
- [ ] **온톨로지**(담을 타입·관계 규격)로 받아 온 사실을 **검증하고 걸러낸다**.
- [ ] **RDF 트리플**(주어-술어-목적어)로 사실을 표현하고 **LPG 와 비교**한다.
- [ ] **rdflib** 로 내 RDF 그래프를 만들고 **SPARQL SELECT** 를 직접 작성해 실행한다.
- [ ] 조건을 거는 문법(`FILTER`·`UNION`·`OPTIONAL`·`NOT EXISTS`)과 결과를 다듬는 문법(`DISTINCT`·`ORDER BY`·`LIMIT`·`COUNT`)을 골라 쓴다.
- [ ] **속성 경로**로 2홉을 한 줄에 적고, `CONSTRUCT` 로 새 그래프를 만들고, `INSERT`·`DELETE` 로 그래프를 고친다.
- [ ] **공개 SPARQL 엔드포인트**에 질의를 보내고 `wd:`·`wdt:`·`?x` 를 읽어 낸다.

## 준비: 공개 지식 그래프에 질의할 준비

오늘 쓸 사실은 **Wikidata** 에서 받아 옵니다. 사람·장소·작품 같은 항목을 1억 개 넘게 담고 있고 그 사이의 사실이 전부 트리플로 들어 있는 **공개 지식 그래프**입니다. 누구나 무료로 **SPARQL 질의**를 보낼 수 있습니다.

### 누가 만들고 관리하나요?

| 누가 | 무엇을 |
|---|---|
| 위키미디어 재단 | 서비스를 운영합니다(위키백과를 만드는 그 재단) |
| 위키미디어 독일 | 소프트웨어를 개발했습니다. 2012년 10월에 문을 열었습니다 |
| 전 세계 편집자와 봇 | 항목과 사실을 직접 채웁니다. 위키백과처럼 **누구나 고칠 수 있습니다** |

데이터는 **CC0**(저작권 포기)로 공개돼 상업적으로도 자유롭게 쓸 수 있습니다(출처 표시는 의무가 아니지만 관례로 밝힙니다).

> 누구나 고칠 수 있다는 점이 **장점이자 한계**입니다. 그래서 오늘 실습에서도 빠진 사실과 지저분한 값을 함께 만나게 됩니다.

### 누가 잘못 고치면 어떻게 되나요?

열려 있으니 실수도 장난도 들어옵니다. Wikidata 는 이를 막기보다 **편집 이력·출처·순찰로 빨리 되돌리는** 쪽으로 설계돼 있습니다.

그래서 쓰는 쪽에서는 **그대로 믿지 않고 한 번 거릅니다.** 2절에서 할 일이 바로 그것입니다. 원본이 무엇을 담고 있든 **내 온톨로지를 통과한 것만** 내 그래프에 넣습니다.

### 오늘 쓰는 라이브러리 두 개

| 라이브러리 | 하는 일 | 오늘 쓰는 곳 |
|---|---|---|
| `SPARQLWrapper` | 원격 SPARQL 엔드포인트에 질의를 보내고 결과를 받아 온다 | 데이터 받아 오기, 5절 |
| `rdflib` | 내 컴퓨터 안에 RDF 그래프를 만들고 SPARQL 을 실행한다 | 3절 잠깐, 4절 본격 |

터미널에서 먼저 설치하세요.

```bash
uv add rdflib SPARQLWrapper
```

### 엔드포인트는 한 곳이 아닙니다

**엔드포인트(endpoint)** 는 질의를 받아 주는 웹 주소입니다. 같은 Wikidata 데이터를 서비스하는 공개 엔드포인트가 여러 곳 있고, 우리는 두 곳을 씁니다.

| 이름 | 주소 | 특징 |
|---|---|---|
| QLever | `https://qlever.cs.uni-freiburg.de/api/wikidata` | 빠르고 요청 제한이 느슨해 수업에서 주로 쓴다 |
| Wikidata 공식 | `https://query.wikidata.org/sparql` | 위키미디어가 직접 운영. 편의 기능이 있지만 요청 제한이 빡빡하다 |

같은 데이터라도 엔드포인트마다 **지원하는 기능과 요청 제한이 다릅니다.** 5절에서 그 차이를 직접 봅니다.

In [ ]:
# [제공 코드] SPARQL 질의를 보낼 준비(실행만 하세요)
from SPARQLWrapper import SPARQLWrapper, JSON

# 질의를 받아 주는 공개 엔드포인트 두 곳. 기본값은 수업에서 주로 쓸 QLever 다
QLEVER = 'https://qlever.cs.uni-freiburg.de/api/wikidata'
WIKIDATA = 'https://query.wikidata.org/sparql'
# 누가 보내는지 밝히는 것이 공개 API 예의다. 헤더는 한글을 못 담으니 영문으로 적는다
USER_AGENT = 'EncoreAICampus-day27/1.0 (classroom practice)'

def run_sparql(query, endpoint=QLEVER):
    """SPARQL 질의문을 보내고 결과 행 리스트를 돌려준다.

    query    : 보낼 SPARQL 질의문
    endpoint : 질의를 받아 줄 주소(생략하면 QLever)
    """
    client = SPARQLWrapper(endpoint, agent=USER_AGENT)
    client.setQuery(query)
    client.setReturnFormat(JSON)   # 결과를 JSON 으로 받는다(파이썬 dict 로 바로 읽힌다)
    client.setTimeout(60)          # 응답이 없으면 60초에 끊는다(수업이 멈추지 않게)
    # 응답 구조: results.bindings 가 결과 행 리스트고, 행마다 SELECT 에 적은 변수 이름이 키다
    return client.query().convert()['results']['bindings']

In [ ]:
# [제공 코드] 연결 확인: 아인슈타인 항목(Q937)의 한국어 이름을 물어본다(실행만 하세요)
# 질의문 읽는 법은 4·5절에서 배운다. 지금은 엔드포인트에 닿는지만 확인한다
hello_query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?name WHERE {
  wd:Q937 rdfs:label ?name .
  FILTER(lang(?name) = "ko")
}
"""
hello_rows = run_sparql(hello_query)
print(hello_rows[0]['name']['value'])   # 출력: 알베르트 아인슈타인 - 이게 찍히면 연결 성공이다

## 데이터 살펴보기

아래 셀들은 **실행만** 하면 됩니다. Wikidata 에서 인물 11명의 **국적**과 **직업** 사실을 받아 옵니다.

질의문 안의 `wdt:P27`(국적)·`wdt:P106`(직업) 같은 기호를 읽는 법은 5절에서 배웁니다. 지금은 받아 온 결과가 **`[주어, 술어, 목적어]` 세 칸짜리 리스트로 쌓인다**는 것만 보면 됩니다.

In [ ]:
# [제공 코드] 오늘 다룰 인물 11명의 Wikidata 항목 번호(실행만 하세요)
# Wikidata 는 이름 대신 번호(Q-id)로 항목을 가리킨다. 동명이인과 섞이지 않게 하려는 것이다
# 키는 등록된 한국어 이름 그대로다(세종은 '조선 세종' 으로 올라 있다)
PEOPLE_QIDS = {
    '알베르트 아인슈타인': 'Q937',
    '아이작 뉴턴': 'Q935',
    '마리 퀴리': 'Q7186',
    '앙투안 라부아지에': 'Q39607',
    '레오나르도 다 빈치': 'Q762',
    '빈센트 반 고흐': 'Q5582',
    '볼프강 아마데우스 모차르트': 'Q254',
    '루트비히 판 베토벤': 'Q255',
    '윌리엄 셰익스피어': 'Q692',
    '찰스 다윈': 'Q1035',
    '조선 세종': 'Q37682',
}
# 번호를 주소 뒤에 붙이면 원본 항목을 웹에서 바로 열어 볼 수 있다
for name in ['알베르트 아인슈타인', '마리 퀴리', '조선 세종']:
    # 출력 3줄: Q937 · Q7186 · Q37682. 주소를 열면 그 인물의 원본 항목이 나온다
    print(f'{name}: https://www.wikidata.org/wiki/{PEOPLE_QIDS[name]}')

In [ ]:
# [제공 코드] 항목 번호 목록으로 사실을 받아 오는 함수(실행만 하세요)
def fetch_facts(qids, prop, rel):
    """항목들의 한 속성 값을 받아 [주어, 술어, 목적어] 리스트로 돌려준다.

    qids : 항목 번호 목록(예: ['Q937', 'Q935'])
    prop : 받아 올 속성 번호(예: 'P27' 국적, 'P106' 직업)
    rel  : 결과의 술어 자리에 넣을 우리말 이름(예: '국적')
    """
    # VALUES 절에 넣을 'wd:Q937 wd:Q935 ...' 문자열을 만든다
    values = ' '.join(f'wd:{q}' for q in qids)
    query = f"""
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?subject ?object WHERE {{
      VALUES ?item {{ {values} }}
      ?item wdt:{prop} ?value .
      # 이름은 언어별로 여러 개다. 한국어 이름만 남긴다(없는 항목은 여기서 빠진다)
      ?item  rdfs:label ?subject .  FILTER(lang(?subject) = "ko")
      ?value rdfs:label ?object .   FILTER(lang(?object) = "ko")
    }}
    ORDER BY ?subject ?object
    """
    rows = run_sparql(query)
    # 술어 자리에는 rel 을 넣는다. 원본 술어는 P27 같은 번호라 읽기 어렵다
    return [[r['subject']['value'], rel, r['object']['value']] for r in rows]

In [ ]:
# [제공 코드] 국적(P27)·직업(P106) 사실을 받아 온다(실행만 하세요)
# 네트워크를 두 번 왕복하므로 몇 초 걸린다
person_ids = list(PEOPLE_QIDS.values())
triples = fetch_facts(person_ids, 'P27', '국적') + fetch_facts(person_ids, 'P106', '직업')
print('받아 온 트리플 수:', len(triples))   # 출력: 100 안팎 - 원본이 갱신되면 조금 달라진다

In [ ]:
# [제공 코드] (이어서)
# 앞 5개만 찍어 트리플 한 줄의 생김새를 눈으로 확인한다
for t in triples[:5]:
    print(t)   # 출력: ['레오나르도 다 빈치', '국적', '피렌체 공화국'] 같은 세 칸 리스트 5줄

In [ ]:
# [제공 코드] (이어서)
# set 으로 중복을 지우고 sorted 로 순서를 고정한다(집합만 쓰면 순서가 들쭉날쭉하다)
print('술어 종류:', sorted({p for s, p, o in triples}))   # 출력: ['국적', '직업']

---
# 1. 지식 그래프란? 사실을 연결로 쌓은 그래프

## 왜 필요할까요?
"아인슈타인"이라는 이름만으로는 아무것도 알 수 없습니다. "아인슈타인 **─국적→** 독일", "아인슈타인 **─직업→** 물리학자"처럼 **다른 개체와 이어질 때** 비로소 지식이 됩니다. 이렇게 사실을 연결로 쌓아 올린 그래프가 **지식 그래프**입니다.

## 비유: 낱말 사전 vs 관계도
낱말 사전은 단어를 하나씩 설명합니다. 지식 그래프는 "누가 무엇과 어떻게 이어지는가"를 그린 **관계도**입니다. 검색엔진이 인물을 검색하면 옆에 뜨는 정보 카드가 바로 지식 그래프에서 나옵니다.

<img src="images/교안02/knowledge_graph_intuition.png" width="820">

> 위 그림은 검색엔진이 보여 주는 카드의 **일반적인 예**입니다. 우리가 방금 받아 온 데이터에는 그중 **국적·직업 두 가지**만 담겨 있습니다.

받아 온 사실은 각각 `[주어, 술어, 목적어]` 리스트입니다. 사람이 읽는 문장으로 풀어 봅니다.

In [ ]:
# 세 칸을 순서대로 풀어 쓰면 그대로 문장이 된다. 트리플 한 줄이 곧 사실 하나다
for subj, pred, obj in triples[:6]:
    print(f'{subj} 의 {pred} 은(는) {obj}')   # 출력: '마리 퀴리 의 국적 은(는) 프랑스' 같은 문장 6줄

In [ ]:
# 인물마다 아는 사실이 몇 개인지 센다. 지식 그래프는 대상마다 담긴 양이 크게 다르다
from collections import Counter

fact_counts = Counter(subj for subj, pred, obj in triples)
for name, count in fact_counts.most_common():
    # 출력 11줄. 여러 분야에 이름이 올라 있는 레오나르도 다 빈치가 가장 많다
    print(name, count)

In [ ]:
# 한 사람의 사실만 골라 본다. 주어를 고정하면 '그 대상에 대해 아는 것' 이 모인다
for subj, pred, obj in triples:
    if '세종' in subj:
        # 출력 4줄: 국적 1개(조선) + 직업 3개(군주·언어학자·정치인)
        print(subj, pred, obj)

> 한 사람에게 사실이 수십 개씩 달려 있습니다. 원본 Wikidata 는 이런 사실을 **수십억 개** 이어 붙인 것이고, 담는 방식은 지금 본 세 칸 그대로입니다.

> **국적이 하나도 없는 사람도 있습니다.** 우리는 한국어 이름이 붙은 값만 받아 오는데, 모차르트의 국적으로 적힌 항목에는 한국어 이름이 없어서 빠졌습니다. 지식 그래프는 사람이 채우는 곳이라 **어느 나라 말로 적혀 있는지도 데이터마다 다릅니다.**

### 🖐️ 함께 따라하기: 한 작품에 대한 사실 모으기

> 데모는 **인물의 국적·직업**이었습니다. 따라하기는 **미술관 소장품**으로 같은 기술을 연습합니다.

먼저 아래 셀을 **실행만** 해서 데이터를 받아 오고, 어떤 사실이 들어 있는지 눈으로 훑어보세요.

In [ ]:
# [제공 코드] 따라하기용 미술관 소장품 트리플(실행만 하세요)
# 데모와 같은 fetch_facts 를 쓰되, 인물 대신 '작품' 항목 다섯 점의 사실을 받아 온다
WORK_QIDS = {
    '모나리자': 'Q12418',
    '별이 빛나는 밤': 'Q45585',
    '진주 귀고리를 한 소녀': 'Q185372',
    '게르니카': 'Q175036',
    '절규': 'Q471379',
}
work_ids = list(WORK_QIDS.values())
# P170 = 창작자(누가 만들었나), P195 = 소장처(어느 미술관 소장품인가)
museum_triples = (fetch_facts(work_ids, 'P170', '창작자')
                  + fetch_facts(work_ids, 'P195', '소장처'))
# 작가의 국적은 작품이 아니라 '작품을 만든 사람' 에 달린 사실이라 질의를 따로 보낸다
artist_query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?subject ?object WHERE {
  VALUES ?work { wd:Q12418 wd:Q45585 wd:Q185372 wd:Q175036 wd:Q471379 }
  ?work wdt:P170 ?person .
  ?person wdt:P27 ?value .
  ?person rdfs:label ?subject . FILTER(lang(?subject) = "ko")
  ?value  rdfs:label ?object .  FILTER(lang(?object) = "ko")
}
ORDER BY ?subject ?object
"""
for row in run_sparql(artist_query):
    museum_triples.append([row['subject']['value'], '국적', row['object']['value']])
print('트리플 수:', len(museum_triples))   # 출력: 15 안팎 (작품 5점 x 2 + 작가 5명의 국적)

In [ ]:
# [제공 코드] (이어서)
# 열다섯 줄이라 전부 찍어 어떤 사실이 들어 있는지 눈으로 확인한다
for t in museum_triples:
    print(t)   # 출력: ['모나리자', '창작자', '레오나르도 다 빈치'] 같은 세 칸 리스트

In [ ]:
# [제공 코드] (이어서)
# 술어 종류: 이 데이터가 담은 사실의 종류
print('술어 종류:', sorted({p for s, p, o in museum_triples}))   # 출력: ['국적', '소장처', '창작자']

이제 `museum_triples` 에서 주어가 **`'모나리자'`** 인 트리플만 골라 리스트 **`mona_facts`** 에 담고 하나씩 출력하세요. 각 트리플의 주어는 첫 칸입니다.

**확인 기준**: 모나리자에 대한 사실 **2개**(창작자·소장처)가 출력되면 맞습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) museum_triples 를 돌며 주어(첫 칸)가 '모나리자' 인 트리플만 mona_facts 에 모은다
# 2) 하나씩 출력한다

### ✅ 바로 확인 퀴즈

**1.** "이름만 있으면 지식이 아니다"라고 말하는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

지식은 개체 하나가 아니라 **개체와 개체 사이의 연결(사실)** 에 담기기 때문입니다. 연결이 있어야 "무엇이 어떠하다"를 알 수 있습니다.

</details>

**2.** 트리플 `['마리 퀴리', '직업', '화학자']` 가 뜻하는 사실을 한 문장으로 쓰면?

<details><summary>정답 보기</summary>

"마리 퀴리의 직업은 화학자이다" 입니다.

</details>

---
# 2. 온톨로지란? 무엇을 담을지 적어 둔 규격

## 온톨로지란?
**온톨로지(ontology)** 는 한 분야에서 쓰는 **말과 그 뜻을 프로그램이 읽을 수 있게 적어 둔 규격**입니다. 보통 세 가지를 적습니다.

| 적어 두는 것 | 뜻 | 우리 데이터에서 |
|---|---|---|
| 타입(클래스) | 이 분야에 어떤 종류가 있는가 | 인물, 국가, 직업 |
| 관계 | 어떤 종류끼리 어떻게 이어질 수 있는가 | 인물에서 국가로 가는 `국적` |
| 제약 | 무엇을 담고 무엇을 막는가 | `국적` 의 목적어는 국가만 |

원래는 철학에서 "무엇이 존재하는가"를 따지던 말(존재론)인데, 데이터에서는 **그 목록과 규칙을 파일로 적어 둔 것**을 가리킵니다. 국제 표준 언어로 **RDFS·OWL** 이 있고, 오늘은 같은 생각을 파이썬 dict 로 작게 만들어 봅니다.

규격이 위, 그 규격을 따르는 실제 사실이 아래라고 보면 둘의 관계가 분명해집니다.

<img src="images/교안02/ontology_layers.png" width="860">

> **관계형 데이터베이스(RDB)의 스키마와 무엇이 다른가요?** 스키마는 "이 열은 정수" 처럼 **저장 형식**을 정합니다. 온톨로지는 "국적은 인물과 국가를 잇는다" 처럼 **뜻**을 정합니다. 뜻이 적혀 있으면 규격 검사뿐 아니라 적어 두지 않은 사실을 끌어내는 일(추론)도 할 수 있습니다. 예를 들어 어떤 값이 `국적` 의 목적어 자리에 왔다면 그 값은 국가라고 판단할 수 있습니다.

## 왜 필요할까요?
받아 온 사실을 그대로 다 쌓으면 그래프가 뒤죽박죽이 됩니다. 실제 원본이 얼마나 잘게 나뉘어 있는지 먼저 봅시다.

In [ ]:
# 받아 온 사실의 '직업' 값이 몇 종류인지 - 원본은 우리가 생각한 것보다 훨씬 잘게 나뉘어 있다
job_values = sorted({obj for subj, pred, obj in triples if pred == '직업'})
print('직업 값 종류:', len(job_values))   # 출력: 56 안팎

In [ ]:
# 어떤 값들인지 앞에서 열두 개만 - 전체를 다 찍으면 화면이 넘친다
# '과학자'·'교수'처럼 넓은 값과 '이론물리학자'처럼 세분된 값이 섞여 있다
print(job_values[:12])

'물리학자'와 '이론물리학자'가 따로 있고, '교수'·'대학 교수'도 따로 있습니다. 이대로 두면 "물리학자는 누구인가"에 답하기 어렵습니다. 그래서 **이 그래프가 담을 것을 미리 못 박습니다.** 그 규격이 온톨로지입니다.

## 문법: 규격은 두 가지를 못 박는다
온톨로지는 **두 가지를 함께** 적습니다. 하나만 있으면 검증이 반쪽이 됩니다.

**1) 관계 규칙**: 관계마다 어떤 타입끼리 이어질 수 있는가

| 관계(술어) | 주어 타입 | 목적어 타입 |
|---|---|---|
| 국적 | 인물 | 국가 |
| 직업 | 인물 | 직업 |

**2) 타입 사전**: 각 타입에 **어떤 값이 속하는가**

| 타입 | 값을 어떻게 정하나 |
|---|---|
| 인물 | 받아 온 사실의 주어 그대로(11명) |
| 국가 | 받아 온 국적 값에서 모으되, `무국적` 처럼 나라 이름이 아닌 것은 뺀다 |
| 직업 | **우리가 손으로 여덟 가지만** 적는다. 원본의 수십 종을 다 담지 않는다 |

관계 규칙만 있으면 `이론물리학자`도 직업이라 통과합니다. **위에서 본 직업 56종이 그대로 들어옵니다.** 값 목록이 있어야 여덟 가지로 좁혀집니다.

두 규격이 무엇을 걸러내는지 그림으로 봅니다.

<img src="images/교안02/ontology_rules.png" width="860">

아래에서 **관계 규칙과 타입 사전을 둘 다** 정의하고, 트리플 하나를 검증하는 함수를 만듭니다.

In [ ]:
# 온톨로지 = 이 그래프에 담을 '타입·관계 규격'
# 관계(술어) -> (주어 타입, 목적어 타입). 여기 적힌 두 관계만 담는다
RELATION_RULES = {
    '국적': ('인물', '국가'),
    '직업': ('인물', '직업'),
}

# 우리 그래프가 담을 직업 여덟 가지. 원본의 수십 종을 다 담지 않고 규격으로 범위를 정한다
VALID_JOBS = {'물리학자', '수학자', '화학자', '화가', '작곡가', '극작가', '생물학자', '군주'}
# 국가는 받아 온 값에서 모은다. 다만 '무국적' 은 나라 이름이 아니므로 규격에서 뺀다
NOT_COUNTRY = {'무국적'}
VALID_COUNTRIES = {obj for subj, pred, obj in triples if pred == '국적'} - NOT_COUNTRY
VALID_PEOPLE = {subj for subj, pred, obj in triples}   # 받아 온 사실의 주어는 모두 인물이다
# 규칙판에 적힌 타입 이름('인물')과 실제 값 집합을 이어 주는 다리
TYPE_DICT = {'인물': VALID_PEOPLE, '국가': VALID_COUNTRIES, '직업': VALID_JOBS}
print('타입별 값 개수:', {t: len(v) for t, v in TYPE_DICT.items()})
# 출력: 인물 11 · 국가 17 안팎 · 직업 8

In [ ]:
# 트리플이 온톨로지에 맞는지 검사: 담기로 한 관계이면서 주어·목적어 타입이 둘 다 맞아야 통과
def is_valid(triple):
    """triple([주어, 술어, 목적어])이 온톨로지 규격에 맞으면 True."""
    subj, pred, obj = triple
    if pred not in RELATION_RULES:
        return False                             # 규격에 없는 관계
    subj_type, obj_type = RELATION_RULES[pred]   # 규칙판에 적힌 두 타입을 꺼내 쓴다
    # and 로 이었으므로 주어 타입·목적어 타입 중 하나만 어긋나도 False 다
    return subj in TYPE_DICT[subj_type] and obj in TYPE_DICT[obj_type]

# 통과 1개와 탈락 4개를 나란히 찍어, 어느 칸이 어긋났을 때 걸리는지 본다(출력 5줄)
print(is_valid(['알베르트 아인슈타인', '직업', '물리학자']))   # 규격에 맞음 -> True
print(is_valid(['알베르트 아인슈타인', '취미', '바이올린']))   # 규격에 없는 관계 -> False
print(is_valid(['알베르트 아인슈타인', '직업', '이론물리학자']))   # 담기로 한 직업이 아님 -> False
print(is_valid(['알베르트 아인슈타인', '국적', '무국적']))   # 목적어가 국가가 아님 -> False
print(is_valid(['독일', '국적', '프랑스']))                    # 주어가 인물이 아님 -> False

> 규칙 `('인물', '국가')` 의 **양쪽을 다 봐야** 합니다. 목적어만 검사하면 "독일의 국적은 프랑스" 처럼 주어가 엉뚱한 사실이 그대로 통과합니다.

이제 규격에 맞는 사실만 남겨 **오늘 다룰 지식 그래프**를 만듭니다.

In [ ]:
# 규격을 통과한 사실만 남긴다. 이것이 3절부터 계속 쓸 우리 지식 그래프다
kg_triples = [t for t in triples if is_valid(t)]
print(f'규격 통과: {len(kg_triples)} / {len(triples)}')   # 출력: 38 / 100 안팎

> **걸러진 사실이 틀린 사실이라는 뜻은 아닙니다.** 아인슈타인이 '이론물리학자'인 것도, 한때 무국적이었던 것도 참입니다. 다만 **우리가 정한 규격 밖**이라 이 그래프에는 담지 않는 것입니다. 지식 그래프를 만들 때 가장 먼저 하는 일이 바로 이 **담을 범위 정하기**입니다.

### ✅ 바로 확인 퀴즈

**1.** 온톨로지가 없으면 지식 그래프에 어떤 문제가 생기나요?

<details><summary>정답 보기</summary>

담을 범위와 관계 이름이 제각각이 됩니다. 값 목록이 없으면 '물리학자'와 '이론물리학자'가 따로 쌓이고, 관계 규칙이 없으면 말이 안 되는 연결(목적어가 국가여야 할 자리에 직업이 오는)이 섞입니다. 둘 다 있어야 그래프가 일관됩니다.

</details>

**2.** 트리플 `['알베르트 아인슈타인', '국적', '무국적']` 과 `['독일', '국적', '프랑스']` 는 각각 왜 검증을 통과하지 못하나요?

<details><summary>정답 보기</summary>

앞은 **목적어** 타입이 어긋납니다(`국적` 의 목적어는 국가여야 하는데 `무국적` 은 나라 이름이 아닙니다). 뒤는 **주어** 타입이 어긋납니다(`독일` 은 인물이 아니라 국가). 규칙 `('인물', '국가')` 의 양쪽을 모두 검사해야 두 경우가 다 걸러집니다.

</details>

---
# 3. RDF 트리플이란? 사실을 적는 웹 표준 형식

## RDF 란?
**RDF(Resource Description Framework, 자원 기술 프레임워크)** 는 웹에 있는 것들을 **`[주어, 술어, 목적어]` 세 칸으로 적자**고 정한 국제 표준입니다(웹 표준을 정하는 기구 W3C 가 제정). 이름 그대로 **자원(Resource)을 설명(Description)하는 틀(Framework)** 이고, 여기서 자원은 사람·장소·작품처럼 **가리킬 수 있는 모든 것**을 뜻합니다.

세 칸에도 각각 이름이 있습니다.

| 칸 | 영어 | 무엇을 담는가 | 예 |
|---|---|---|---|
| 주어 | subject | 무엇에 대한 사실인가 | 알베르트 아인슈타인 |
| 술어 | predicate | 어떤 관계·성질인가 | 국적 |
| 목적어 | object | 그 값은 무엇인가 | 독일 |

> **목적어 자리에는 두 가지가 옵니다.** 다른 개체(예: 국가 '독일')를 가리키면 그 개체의 **주소**를 적고, 이름·숫자·날짜처럼 그냥 값이면 **값 그대로**(RDF 에서 리터럴이라 부릅니다) 적습니다. 오늘 다루는 사실은 전부 개체끼리 이어진 것이라 양쪽이 다 주소입니다.

표준에서는 자원을 이름 대신 **주소(URI)** 로 가리킵니다. 이름이 같은 다른 사람과 섞이지 않게 하려는 것입니다. 우리가 받아 온 사실도 원본에서는 주소로 적혀 있습니다.

In [ ]:
# 우리는 읽기 쉬우라고 이름으로 받아 왔지만, 원본이 쓰는 식별자는 주소(URI)다
einstein_uri = 'http://www.wikidata.org/entity/' + PEOPLE_QIDS['알베르트 아인슈타인']
print(einstein_uri)   # 출력: http://www.wikidata.org/entity/Q937

리터럴은 값에 **데이터형**이나 **언어**를 붙여 둘 수 있습니다. 어떻게 적히는지 눈으로 보고 넘어갑니다. 여기 쓰는 `rdflib` 은 **4절에서 제대로 배웁니다**(지금은 실행만 하세요).

### RDF 를 글로 적는 표기: Turtle

트리플을 파일로 주고받으려면 **글로 적는 방법**이 필요합니다. 가장 널리 쓰는 것이 **Turtle** 이고, 사람이 읽기 좋게 세 가지를 줄여 씁니다.

- 긴 주소에 **이름표**를 붙여 줄인다. `bind('ex', ...)` 로 붙인 이름표가 `ex:모나리자` 의 `ex:` 다(이름표를 `kg` 로 붙였다면 `kg:모나리자` 가 된다)
- **같은 주어**가 이어지면 세미콜론(`;`)으로 묶어 주어를 한 번만 적는다
- **같은 술어**가 이어지면 쉼표(`,`)로 묶어 술어를 한 번만 적는다

그래서 아래 셀은 트리플이 **두 개**인데 주어 `ex:모나리자` 는 **한 번만** 나옵니다.

| `format` | 어떻게 적히나 |
|---|---|
| `turtle` | 사람이 읽기 좋게 압축. 기본으로 쓴다 |
| `nt` | 압축 없이 **한 줄 = 트리플 하나**(4절에서 본다) |
| `json-ld` | JSON 형태. 프로그램끼리 주고받을 때 |

`format` 을 아예 생략하면 기본값인 `turtle` 로 적힙니다.

In [ ]:
# [제공 코드] 리터럴이 어떻게 적히는지만 보고 넘어갑니다(4절에서 rdflib 을 제대로 배웁니다)
from rdflib import Graph, Literal, Namespace

EX = Namespace('http://example.org/kg/')
literal_demo = Graph()
literal_demo.bind('ex', EX)
literal_demo.add((EX['모나리자'], EX['제작연도'], Literal(1503)))          # 숫자 값
literal_demo.add((EX['모나리자'], EX['별명'], Literal('라 조콘다', lang='ko')))   # 한국어 값
# 출력: 주어는 한 번만 나오고, 숫자는 따옴표 없이, 언어가 붙은 값은 "..."@ko 로 적힌다
print(literal_demo.serialize(format='turtle'))

## 왜 필요할까요?
이 형식이 국제 표준이라 **서로 다른 지식 그래프를 이어 붙일 때 공통 언어**가 됩니다. 기관마다 제각각인 표 구조를 맞추는 대신, 같은 세 칸으로 적어 두면 그대로 합칠 수 있습니다.

## 문법: RDF 와 LPG 는 같은 그래프의 두 표기
지난 시간의 **LPG** 도, 오늘의 **RDF** 도 결국 노드와 관계로 이루어진 그래프입니다. 담는 방식이 다를 뿐입니다.

| 항목 | RDF 트리플 | LPG(속성 그래프) |
|---|---|---|
| 기본 단위 | 트리플 (주어-술어-목적어) | 노드·관계·속성 |
| 속성 표현 | 모든 것이 트리플(속성도 트리플) | 노드·관계 안에 key-value |
| 방향 | 주어 → 목적어 | 출발 → 도착 |
| 강점 | 표준·데이터 통합 | 속성 다루기·쿼리 편의 |

그림으로 보면 같은 사실이 두 표기에서 어떻게 생겼는지가 한눈에 들어옵니다.

<img src="images/교안02/rdf_vs_lpg.png" width="760">

RDF 트리플을 **LPG 스타일 노드·관계**로 옮겨 보면, 둘이 같은 사실을 담고 있음이 보입니다. 어떤 타입으로 옮길지는 **온톨로지 규칙**이 알려 주므로, 규칙 사전을 함수에 함께 넘깁니다.

In [ ]:
# RDF 트리플 하나를 LPG 스타일(노드 두 개 + 방향 관계)로 옮긴다
def triple_to_lpg(triple, rules):
    """트리플을 (노드 A, 노드 B, 관계) 세 값으로 바꿔 돌려준다.

    triple : 옮길 [주어, 술어, 목적어]
    rules  : 온톨로지 규칙 사전(술어 -> (주어 타입, 목적어 타입))
    """
    subj, pred, obj = triple
    # 트리플 자체에는 타입이 없다. 노드에 붙일 레이블은 규칙판에서 가져온다
    subj_type, obj_type = rules[pred]
    node_a = {'id': subj, 'label': subj_type}
    node_b = {'id': obj, 'label': obj_type}
    edge = (subj, pred, obj)               # 방향: 주어 -> 목적어
    return node_a, node_b, edge

a, b, e = triple_to_lpg(['알베르트 아인슈타인', '국적', '독일'], RELATION_RULES)
print('노드 A:', a)   # 출력: {'id': '알베르트 아인슈타인', 'label': '인물'}
print('노드 B:', b)   # 출력: {'id': '독일', 'label': '국가'}
print('관계  :', e)   # 출력: ('알베르트 아인슈타인', '국적', '독일') - 트리플 한 줄이 노드 2 + 관계 1 이 됐다

> 같은 사실이 RDF 에서는 트리플 한 줄, LPG 에서는 노드 두 개와 방향 관계 하나가 됩니다. 표기가 다를 뿐 **구조는 동일**합니다.

### ✅ 바로 확인 퀴즈

**1.** RDF 가 "웹의 표준"이라는 점이 실무에서 주는 이점은 무엇인가요?

<details><summary>정답 보기</summary>

서로 다른 곳에서 만든 지식 그래프라도 **같은 형식(트리플)** 이라 이어 붙이고 통합하기 쉽습니다.

</details>

**2.** RDF 트리플과 LPG 는 근본적으로 무엇이 같나요?

<details><summary>정답 보기</summary>

둘 다 **노드와 방향 관계로 이루어진 그래프**입니다. 사실을 담는 표기 방식만 다릅니다.

</details>

---
# 4. SPARQL이란? RDF에 묻는 표준 쿼리 언어

## 왜 필요할까요?
"물리학자인 사람은 **누구누구**인가?"처럼, 트리플의 한 칸을 **빈칸(변수)** 으로 두고 그 자리에 들어갈 값을 모두 찾고 싶을 때가 있습니다. 반복문으로도 되지만, 사실이 수천만 개가 되면 "어떻게 찾을지"를 일일이 적기 어렵습니다. **SPARQL** 은 "무엇을 찾을지"만 적으면 되는 표준 쿼리 언어입니다. **관계형 데이터베이스(RDB)에 SQL 이 있다면, RDF 에는 SPARQL 이 있습니다.**

## rdflib: 내 컴퓨터 안의 RDF 그래프
**rdflib** 는 파이썬에서 RDF 를 다루는 표준 라이브러리입니다(3절에서 리터럴을 찍어 볼 때 잠깐 썼던 그것입니다). 트리플을 담는 `Graph` 를 만들고, **그 위에 SPARQL 질의를 그대로 실행**할 수 있습니다. 원격 엔드포인트에 보내기 전에, 먼저 우리 손안의 그래프로 문법을 익힙니다.

| 하는 일 | rdflib 표현 |
|---|---|
| 그래프 만들기 | `graph = Graph()` |
| 주소 앞부분 정하기 | `EX = Namespace('http://example.org/kg/')` |
| 트리플 넣기 | `graph.add((주어, 술어, 목적어))` |
| SPARQL 실행 | `graph.query(질의문)` |
| Turtle 로 출력 | `graph.serialize()` |

`graph.add` 의 세 칸에는 **파이썬 문자열을 그대로 넣을 수 없습니다**(rdflib 이 거부합니다). 개체는 주소로, 값은 값으로 감싸 넣습니다.

| 넣을 것 | 쓰는 법 | 뜻 |
|---|---|---|
| 개체 | `EX['독일']` | `http://example.org/kg/독일` 주소를 만드는 짧은 표기 |
| 값(리터럴) | `Literal(1879)` | 숫자·이름·날짜처럼 가리킬 대상이 없는 값 |

먼저 2절에서 걸러 낸 `kg_triples` 를 진짜 RDF 그래프로 옮깁니다. 우리 사실은 개체끼리 이어진 것이라 세 칸이 모두 주소입니다.

In [ ]:
# 3절에서 만든 EX(주소 앞부분)를 그대로 쓴다
# example.org 는 예제 전용으로 비워 둔 도메인이라 누구나 마음대로 써도 된다
graph = Graph()
graph.bind('ex', EX)   # 이 주소에 'ex' 라는 짧은 이름표를 붙인다(질의문에서 ex: 로 쓴다)

for subject, predicate, obj in kg_triples:
    # URI 에는 공백을 쓸 수 없다. 이름의 공백을 밑줄로 바꿔 주소를 만든다
    graph.add((EX[subject.replace(' ', '_')], EX[predicate], EX[obj.replace(' ', '_')]))
print('그래프에 담긴 트리플 수:', len(graph))   # 출력: 38 안팎 - kg_triples 와 같은 수

In [ ]:
# format 을 생략하면 Turtle 로 적어 준다(3절에서 본 그 표기다)
print(graph.serialize())

> 이름의 공백을 밑줄로 바꾼 이유는 **URI 에 공백을 쓸 수 없기 때문**입니다. 주소는 웹에서 그대로 주고받는 문자열이라 띄어쓰기가 허용되지 않습니다.

방금 찍힌 것이 3절에서 본 **Turtle** 입니다. 사람이 여럿이라 길게 나왔으니, 압축이 어떻게 일어나는지는 **한 사람 것만 따로 떼어** 보면 또렷합니다.

In [ ]:
# 아이작 뉴턴에 대한 사실만 작은 그래프에 담아 Turtle 표기로 찍어 본다
newton_graph = Graph()
newton_graph.bind('ex', EX)
for subject, predicate, obj in kg_triples:
    if subject == '아이작 뉴턴':
        newton_graph.add((EX[subject.replace(' ', '_')], EX[predicate],
                          EX[obj.replace(' ', '_')]))
# 3절에서 본 압축 때문에 트리플 3개가 2줄로 나온다(국적 두 개가 쉼표로 묶인다)
print(newton_graph.serialize(format='turtle'))

> 3절에서 본 압축 그대로입니다. 뉴턴의 국적이 두 개라 쉼표(`,`)로 묶여 트리플 세 개가 두 줄이 됐습니다. 압축 없이 보려면 `format` 을 `nt` 로 바꿉니다.

In [ ]:
# 같은 그래프를 N-Triples 로 찍는다. 주소를 줄이지도, 같은 주어를 묶지도 않는다
# 줄은 길어지지만 '한 줄 = 트리플 하나' 라 트리플의 경계가 또렷하게 보인다
print(newton_graph.serialize(format='nt'))   # 출력 3줄

## 문법: SELECT 와 WHERE
SPARQL 질의문은 크게 세 부분입니다.

| 조각 | 하는 일 |
|---|---|
| `PREFIX ex: <http://example.org/kg/>` | 긴 주소에 짧은 이름표를 붙인다. 이후 `ex:직업` 으로 쓸 수 있다 |
| `SELECT ?person` | 결과 표에 어떤 칸을 담을지 고른다. `?` 로 시작하는 것이 **변수(빈칸)** 다 |
| `WHERE { ... }` | 찾을 **트리플 패턴**을 적는다. 한 줄이 트리플 하나이고 마침표로 끝난다 |

`PREFIX` 는 **그 질의문에서 실제로 쓰는 이름표만** 적으면 됩니다. 아래 질의는 `ex:` 하나만 쓰므로 한 줄이면 충분합니다.

패턴 `?person ex:직업 ex:물리학자 .` 는 "주어는 아무나(`?person`), 술어는 직업, 목적어는 물리학자"라는 뜻입니다. 이 패턴에 맞는 트리플을 모두 찾아 `?person` 자리에 들어갈 값을 돌려줍니다.

<img src="images/교안02/sparql_binding.png" width="760">

In [ ]:
# 진짜 SPARQL 질의문: 직업이 물리학자인 사람은 누구인가
# 질의문마다 앞에 붙일 PREFIX 는 한 번 만들어 두고 이어 붙여 쓴다
QUERY_PREFIX = 'PREFIX ex: <http://example.org/kg/>\n'

physicist_query = QUERY_PREFIX + """
SELECT ?person WHERE {
  ?person ex:직업 ex:물리학자 .
}
ORDER BY ?person
"""
rows = list(graph.query(physicist_query))   # 결과를 한 번 받아 두고 두 가지로 찍어 본다

# 먼저 돌려받은 그대로 본다. 한 행은 튜플이고, 값은 문자열이 아니라 URI 다
for row in rows:
    print(row)

In [ ]:
# 이제 사람이 읽게 다듬는다. 주소의 마지막 칸만 잘라 내고 밑줄을 공백으로 되돌린다
# row.person 은 SELECT 에 적은 변수 이름(?person)으로 그 칸을 꺼내는 것이다
for row in rows:
    print(str(row.person).split('/')[-1].replace('_', ' '))
# 출력 4줄: 레오나르도 다 빈치·마리 퀴리·알베르트 아인슈타인·앙투안 라부아지에

> **아이작 뉴턴이 없습니다.** Wikidata 의 뉴턴 항목에는 직업으로 수학자·철학자가 적혀 있고 물리학자는 적혀 있지 않습니다. 지식 그래프는 사람이 채우는 곳이라 **빠진 사실**이 있습니다. 쿼리는 "적혀 있는 것"만 찾을 수 있으므로, 결과가 비면 먼저 **원본에 그 사실이 있는지** 확인해야 합니다.

### 패턴 두 개를 같은 변수로 잇기

"**물리학자인 사람**의 **국적**은?" 은 패턴 하나로는 답할 수 없습니다. 패턴을 두 줄 쓰고 **같은 변수 이름 `?person`** 을 양쪽에 두면, SPARQL 이 "두 패턴을 모두 만족하는 값"만 남겨 줍니다. 이것이 **조인**입니다.

In [ ]:
# 패턴 두 줄에 같은 ?person 을 쓴다 -> 두 조건을 모두 만족하는 사람만 남는다
join_query = QUERY_PREFIX + """
SELECT ?person ?country WHERE {
  ?person ex:직업 ex:물리학자 .
  ?person ex:국적 ?country .
}
ORDER BY ?person ?country
"""
for row in graph.query(join_query):
    person = str(row.person).split('/')[-1].replace('_', ' ')
    country = str(row.country).split('/')[-1].replace('_', ' ')
    # 출력 11줄 안팎. 아인슈타인처럼 국적이 여럿이면 사람 하나가 여러 줄로 나온다
    print(person, '->', country)

> 물리학자는 네 명인데 결과는 열 줄이 넘습니다. **국적이 여럿인 사람**은 그 수만큼 줄이 늘어나기 때문입니다. 조인은 "양쪽을 다 만족하는 조합"을 모두 돌려줍니다.

### 온톨로지를 RDF 로 적기: `rdf:` 와 `rdfs:`

2절에서 타입 사전(`TYPE_DICT`)과 관계 규칙(`RELATION_RULES`)을 파이썬으로 적어 두었습니다. RDF 에는 그것을 적는 **표준 낱말 두 벌**이 있습니다. 지금까지 쓴 `ex:` 는 우리가 지어낸 이름표지만, 이 둘은 **전 세계가 같은 뜻으로 쓰는** 이름표입니다.

| 이름표 | 정식 이름 | 담는 것 |
|---|---|---|
| `rdf:` | RDF | **개체 하나**에 대해 적는 기본 낱말. 대표가 `rdf:type`(`a`) |
| `rdfs:` | RDF Schema | **타입과 술어 자체**를 정의하는 낱말 |

둘의 경계는 **개체를 말하느냐, 타입을 말하느냐**입니다. `rdf:type` 은 "마리 퀴리는 인물이다" 처럼 **개체 하나가 어느 타입에 속하는지**만 적습니다. "인물이라는 타입이 있다", "이론물리학자는 물리학자의 하위다", "국적이라는 술어는 인물과 국가를 잇는다" 처럼 **타입과 술어 자체를 규정하는 일**은 전부 `rdfs:` 가 맡습니다.

**`rdf:` 에서 실제로 쓰는 것은 사실상 `rdf:type` 하나입니다.** 다른 낱말도 있지만 목록·순서를 적는 특수한 용도라 거의 나오지 않습니다. 게다가 SPARQL 과 Turtle 이 **`a` 한 글자**로 줄여 주기 때문에, `rdf:` 라고 적을 일조차 드뭅니다.

- `ex:마리_퀴리 a ex:인물 .` : 마리 퀴리는 인물이다(**개체**의 소속)

**반대로 `rdfs:` 는 여러 개를 씁니다.** 자주 쓰는 다섯은 이렇습니다.

| 낱말 | 뜻 | 예 |
|---|---|---|
| `rdfs:Class` | **타입 자체**를 선언한다 | `ex:인물 a rdfs:Class` |
| `rdfs:subClassOf` | 타입끼리의 **상하 관계** | `ex:이론물리학자 rdfs:subClassOf ex:물리학자` |
| `rdfs:domain` | 그 술어의 **주어**가 어떤 타입이어야 하는지 | `ex:국적 rdfs:domain ex:인물` |
| `rdfs:range` | 그 술어의 **목적어**가 어떤 타입이어야 하는지 | `ex:국적 rdfs:range ex:국가` |
| `rdfs:label` | 사람이 읽는 **이름** | `wd:Q937 rdfs:label "알베르트 아인슈타인"@ko` |

- 앞의 넷이 2절에서 파이썬으로 적은 **온톨로지 그 자체**입니다. `TYPE_DICT` 의 타입 이름이 `rdfs:Class`, `RELATION_RULES` 의 두 타입이 `rdfs:domain`·`rdfs:range` 에 해당합니다.
- 다만 `rdfs:Class` 는 **적지 않아도 됩니다.** `ex:마리_퀴리 a ex:인물` 만 있으면 타입은 그대로 저장되고 질의 결과도 같습니다. 그래프 안에 스키마를 **문서로 남기려는 선언**이라, 공개된 온톨로지 파일에는 거의 늘 있지만 우리처럼 질의만 할 때는 생략합니다.
- `rdfs:label` 은 **이미 쓰고 있습니다.** 준비 셀이 Wikidata 에서 한국어 이름을 가져올 때 쓴 그 술어입니다.
- `rdfs:domain`·`rdfs:range` 는 2절 `RELATION_RULES` 에 적은 **주어 타입·목적어 타입**을 표준으로 옮긴 것입니다.
- `rdfs:subClassOf` 는 5절에서 볼 Wikidata 의 `wdt:P279`(상위 개념)에 해당합니다.

> 다만 **RDFS 는 검사기가 아닙니다.** `rdfs:domain` 을 적어 두어도 규칙을 어긴 트리플이 자동으로 걸러지지는 않습니다(오히려 "그러니 이 주어는 인물이겠구나" 하고 **추론**하는 쪽입니다). 2절에서 `is_valid` 로 직접 검사해 걸러 낸 이유입니다.

In [ ]:
from rdflib import RDF, RDFS

# 1) 2절의 타입 사전을 옮겨 '누가 그 타입에 속하는지' 를 적는다. 술어는 표준 rdf:type 이다
for type_name, members in TYPE_DICT.items():
    for member in members:
        graph.add((EX[member.replace(' ', '_')], RDF.type, EX[type_name]))

# 2) 타입끼리의 상하 관계도 한 줄 적어 둔다(계층은 5절에서 Wikidata 로 제대로 본다)
graph.add((EX['이론물리학자'], RDFS.subClassOf, EX['물리학자']))

# 3) 2절 RELATION_RULES 의 '국적': ('인물', '국가') 규칙을 표준 술어로 옮긴다
graph.add((EX['국적'], RDFS.domain, EX['인물']))   # 주어는 인물이어야 한다
graph.add((EX['국적'], RDFS.range, EX['국가']))    # 목적어는 국가여야 한다
print('타입·규칙까지 담은 트리플 수:', len(graph))   # 출력: 77 안팎

In [ ]:
# format 은 적어도 되고 앞에서처럼 생략해도 된다. 결과는 같다
print(graph.serialize(format='turtle'))

> 사람마다 첫 줄에 **`a ex:인물`**(개체의 소속)이 붙었고, `ex:국적 rdfs:domain ex:인물 ; rdfs:range ex:국가 .` 처럼 **술어 자체의 규칙**을 적은 블록도 새로 생겼습니다. 맨 위 이름표에도 `@prefix rdfs:` 한 줄이 늘었습니다.

이제 `a` 로 질의해 봅니다. **`a` 는 SPARQL 이 미리 아는 키워드라 `PREFIX` 가 필요 없습니다.** 그래서 앞에서 만든 `QUERY_PREFIX` 를 그대로 씁니다. 반대로 질의문에 `rdfs:subClassOf` 처럼 **이름표를 글자로 적는 순간**에는 `PREFIX rdfs: <...>` 한 줄을 더해야 합니다(5절의 표준 SPARQL 판에서 `rdfs:label` 을 쓸 때 그 줄이 나옵니다).

In [ ]:
# a 는 rdf:type 의 약어다. '인물인 것' 을 이름 순으로 세 명만 본다
type_query = QUERY_PREFIX + """
SELECT ?person WHERE {
  ?person a ex:인물 .
}
ORDER BY ?person
LIMIT 3
"""
for row in graph.query(type_query):
    # 출력 3줄. LIMIT 로 앞에서 세 명만 잘랐다
    print(str(row.person).split('/')[-1].replace('_', ' '))

In [ ]:
# 타입 이름만 바꾸면 그대로 쓸 수 있다. '직업인 것' 은 2절에서 규격으로 정한 여덟 가지다
job_query = QUERY_PREFIX + """
SELECT ?job WHERE {
  ?job a ex:직업 .
}
ORDER BY ?job
"""
for row in graph.query(job_query):
    # 출력 8줄: 군주·극작가·물리학자·생물학자·수학자·작곡가·화가·화학자
    print(str(row.job).split('/')[-1])

이제 **미술관 소장품 그래프**도 함께 만들어 둡니다. 앞으로 나올 문법 예시와 따라하기가 이 그래프를 씁니다(작품 → 작가 → 국가로 2홉 이어지는 구조라 경로 문법을 보기 좋습니다).

In [ ]:
# [제공 코드] 미술관 트리플을 rdflib 그래프로 옮긴다(실행만 하세요)
museum_graph = Graph()
museum_graph.bind('ex', EX)
for subject, predicate, obj in museum_triples:
    museum_graph.add((EX[subject.replace(' ', '_')], EX[predicate],
                      EX[obj.replace(' ', '_')]))
print('그래프에 담긴 트리플 수:', len(museum_graph))   # 출력: 15 안팎

### 문법 더 보기: 조건 걸기와 결과 다듬기

지금까지는 패턴만 적었습니다. 실제 질의에는 **조건**과 **결과 다듬기**가 함께 붙습니다. 자주 쓰는 것만 모았고, 아래에서 하나씩 실행해 봅니다.

- `FILTER(...)` : 조건에 맞는 것만 남긴다. 비교(`=`, `!=`, `>`)를 쓴다
- `UNION` : 두 패턴 중 **하나라도** 맞으면 남긴다
- `OPTIONAL { ... }` : 있으면 채우고 **없으면 빈칸**으로 둔다(행이 사라지지 않는다)
- `FILTER NOT EXISTS { ... }` : 그 패턴이 **없는** 것만 남긴다
- `DISTINCT` : 똑같은 행이 여러 번 나오면 하나로 줄인다
- `ORDER BY` · `LIMIT` : 정렬하고 개수를 자른다
- `COUNT` · `GROUP BY` : 묶어서 센다(맨 아래 응용에서 직접 씁니다)
- `BIND(식 AS ?새변수)` : 계산한 값을 새 변수에 담는다
- `CONTAINS`·`REGEX` : 글자가 들어 있는지, 정규식에 맞는지로 거른다
- `ASK { ... }` : 그런 사실이 있는지 **참/거짓**만 돌려준다

In [ ]:
# UNION: 두 패턴 중 하나라도 맞으면 남긴다 - 화가이거나 작곡가인 사람
union_query = QUERY_PREFIX + """
SELECT ?person WHERE {
  { ?person ex:직업 ex:화가 . }
  UNION
  { ?person ex:직업 ex:작곡가 . }
}
ORDER BY ?person
"""
for row in graph.query(union_query):
    # 출력 5줄. 화가이면서 작곡가인 레오나르도 다 빈치가 두 번 나온다
    print(str(row.person).split('/')[-1].replace('_', ' '))

> 같은 사람이 두 번 나왔습니다. 두 패턴에 **모두** 걸렸기 때문입니다. 중복이 싫으면 `SELECT` 뒤에 `DISTINCT` 를 붙입니다.

In [ ]:
# DISTINCT: 똑같은 행이 여러 번 나오면 하나로 줄인다
distinct_query = QUERY_PREFIX + """
SELECT DISTINCT ?person WHERE {
  { ?person ex:직업 ex:화가 . }
  UNION
  { ?person ex:직업 ex:작곡가 . }
}
ORDER BY ?person
"""
for row in graph.query(distinct_query):
    # 출력 4줄. 두 번 나오던 레오나르도 다 빈치가 한 줄로 줄었다
    print(str(row.person).split('/')[-1].replace('_', ' '))

In [ ]:
# OPTIONAL: 있으면 채우고 없으면 빈칸. 조건이 아니라서 행이 사라지지 않는다
optional_query = QUERY_PREFIX + """
SELECT ?person ?country WHERE {
  ?person ex:직업 ex:작곡가 .
  OPTIONAL { ?person ex:국적 ?country . }
}
ORDER BY ?person
"""
optional_rows = list(graph.query(optional_query))

# 빈칸이 어떻게 오는지 원형 그대로 본다. 국적이 없는 모차르트는 둘째 칸이 None 이다
for row in optional_rows:
    print(row)

In [ ]:
# 그 None 을 사람이 읽을 말로 바꿔 준다
for row in optional_rows:
    person = str(row.person).split('/')[-1].replace('_', ' ')
    country = '(없음)'
    if row.country:                      # 국적이 없는 행은 None 이 온다
        country = str(row.country).split('/')[-1].replace('_', ' ')
    # 출력 4줄. 국적이 없는 모차르트도 (없음) 으로 남는다
    print(person, '->', country)

> `OPTIONAL` 을 빼고 `?person ex:국적 ?country .` 를 그냥 적었다면 **모차르트는 아예 사라집니다.** 패턴을 나란히 적는 것은 "둘 다 있어야 한다"는 뜻이기 때문입니다. "있으면 보여 주고 없어도 빼지 마라"가 필요할 때 `OPTIONAL` 을 씁니다.

In [ ]:
# FILTER NOT EXISTS: 그 패턴이 '없는' 것만 남긴다
not_exists_query = QUERY_PREFIX + """
SELECT ?person WHERE {
  ?person ex:직업 ex:물리학자 .
  FILTER NOT EXISTS { ?person ex:직업 ex:화학자 . }
}
ORDER BY ?person
"""
for row in graph.query(not_exists_query):
    # 출력 1줄: 알베르트 아인슈타인. 나머지 물리학자 셋은 화학자로도 적혀 있다
    print(str(row.person).split('/')[-1].replace('_', ' '))

In [ ]:
# ASK: 행이 아니라 참/거짓만 돌려준다. '있나 없나' 만 궁금할 때 쓴다
ask_newton = QUERY_PREFIX + 'ASK { ex:아이작_뉴턴 ex:직업 ex:물리학자 . }'
ask_curie = QUERY_PREFIX + 'ASK { ex:마리_퀴리 ex:직업 ex:물리학자 . }'
print(bool(graph.query(ask_newton)))   # 출력: False - 원본에 그 사실이 없다
print(bool(graph.query(ask_curie)))    # 출력: True

In [ ]:
# BIND: 계산한 값을 새 변수에 담는다. 여기서는 URI 를 사람이 읽는 이름으로 바꿔 둔다
# STRAFTER(문자열, 'kg/') 은 'kg/' 뒤쪽만 잘라 내고, REPLACE 는 밑줄을 공백으로 바꾼다
# CONTAINS 로 이름에 특정 글자가 든 사람만 남긴다
bind_query = QUERY_PREFIX + """
SELECT ?name WHERE {
  ?person a ex:인물 .
  BIND(REPLACE(STRAFTER(STR(?person), "kg/"), "_", " ") AS ?name)
  FILTER(CONTAINS(?name, "퀴리"))
}
"""
for row in graph.query(bind_query):
    # 출력 1줄: 마리 퀴리. 파이썬에서 자르지 않고 질의 안에서 이름을 만들었다
    print(row.name)

In [ ]:
# REGEX: 정규식으로 거른다. '알' 로 시작하는 이름만 남긴다(^ 는 문자열 시작)
regex_query = QUERY_PREFIX + """
SELECT ?name WHERE {
  ?person a ex:인물 .
  BIND(REPLACE(STRAFTER(STR(?person), "kg/"), "_", " ") AS ?name)
  FILTER(REGEX(?name, "^알"))
}
"""
for row in graph.query(regex_query):
    # 출력 1줄: 알베르트 아인슈타인
    print(row.name)

### 그래프다운 문법: 속성 경로

"작품의 작가가 어느 나라 사람인가"는 **2홉**입니다. 관계를 한 번 따라가면 1홉이라고 불렀죠(교안_01 4절). 작품에서 작가로 1홉, 작가에서 국가로 다시 1홉이라 모두 2홉입니다. 지금까지는 이걸 패턴 두 줄로 쓰고 같은 변수로 이었습니다. SPARQL 에는 **여러 홉을 한 줄로 적는** 문법이 있습니다. 이것을 **속성 경로(property path)** 라 부르고, 그래프 질의를 짧고 또렷하게 만들어 줍니다.

- `술어1/술어2` : 술어를 슬래시로 이어 **2홉을 한 줄로** 적는다(작품 → 작가 → 국가)
- `술어*` · `술어+` : 같은 술어를 **몇 홉이든** 따라간다(계층을 타고 올라갈 때. 5절에서 씁니다)

**같은 질문을 두 번 적어 견줍니다.** 먼저 지금까지 하던 대로 두 줄로, 그다음 경로로 한 줄에.

In [ ]:
# 1홉 두 번: 중간에 거친 작가를 ?artist 변수로 두고, 두 패턴에 같은 이름을 써서 잇는다(조인)
two_step_query = QUERY_PREFIX + """
SELECT ?work ?country WHERE {
  ?work ex:창작자 ?artist .
  ?artist ex:국적 ?country .
}
ORDER BY ?work
"""
two_step_rows = list(museum_graph.query(two_step_query))
for row in two_step_rows:
    work = str(row.work).split('/')[-1].replace('_', ' ')
    country = str(row.country).split('/')[-1].replace('_', ' ')
    # 출력 5줄. 작가는 잇는 데만 쓰고 SELECT 에는 안 적었다
    print(work, '->', country)

In [ ]:
# 경로 / : 같은 질문을 한 줄로. 작품에서 창작자로 1홉, 거기서 국적으로 또 1홉
# 중간에 거쳐 간 작가는 아예 변수로 두지 않는다(작가도 보려면 위처럼 두 줄로 써야 한다)
path_query = QUERY_PREFIX + """
SELECT ?work ?country WHERE {
  ?work ex:창작자/ex:국적 ?country .
}
ORDER BY ?work
"""
path_rows = list(museum_graph.query(path_query))
for row in path_rows:
    work = str(row.work).split('/')[-1].replace('_', ' ')
    country = str(row.country).split('/')[-1].replace('_', ' ')
    # 출력 5줄. 위 결과와 한 줄도 다르지 않다
    print(work, '->', country)

In [ ]:
# 눈으로 견주지 말고 직접 대조한다. 적는 법만 다르고 답은 같다
print(set(two_step_rows) == set(path_rows))   # 출력: True

### 결과가 표가 아니라 그래프일 때: CONSTRUCT

`SELECT` 은 **표**를 돌려줍니다. RDF 에는 질의 결과로 **새 그래프**를 만드는 `CONSTRUCT` 가 따로 있습니다. "찾은 것을 이런 모양의 트리플로 다시 적어 달라"는 뜻이라, 흩어진 사실을 **요약한 그래프로 갈아 끼울 때** 씁니다.

In [ ]:
# CONSTRUCT: WHERE 로 찾은 것을 중괄호 안의 모양으로 다시 적어 새 그래프를 만든다
construct_query = QUERY_PREFIX + """
CONSTRUCT { ?work ex:작가국적 ?country . }
WHERE     { ?work ex:창작자/ex:국적 ?country . }
"""
summary_graph = Graph()
summary_graph.bind('ex', EX)
for fact in museum_graph.query(construct_query):
    summary_graph.add(fact)   # 돌려주는 것이 행이 아니라 트리플이다
print('새 그래프 트리플 수:', len(summary_graph))   # 출력: 5

In [ ]:
# 2홉(창작자 -> 국적)이 1홉(작가국적)으로 접힌 그래프가 만들어졌다
print(summary_graph.serialize(format='turtle'))

### 그래프를 고치는 질의: 넣기·지우기·바꾸기

지금까지는 **읽기**만 했습니다. SPARQL 에는 그래프를 **고치는** 질의도 있습니다. 관계형 데이터베이스에 `SELECT` 말고 `INSERT`·`DELETE` 가 있는 것과 같습니다. rdflib 에서는 `graph.query` 가 아니라 **`graph.update`** 로 보냅니다.

다만 **`UPDATE` 라는 문장은 없습니다.** RDF 에는 표의 칸 같은 것이 없고 트리플이 최소 단위라, "값을 바꾼다"는 곧 **옛 트리플을 지우고 새 트리플을 넣는 것**입니다. 그래서 고치는 문법은 세 가지입니다.

| 문법 | 하는 일 |
|---|---|
| `INSERT DATA` · `DELETE DATA` | 적어 준 트리플 **그대로** 넣고 뺀다 |
| `INSERT ... WHERE` · `DELETE ... WHERE` | 조건에 맞는 것을 **찾아서** 한꺼번에 |
| `DELETE ... INSERT ... WHERE` | 지우기와 넣기를 한 덩어리로 보낸다. 이것이 **수정** |

수정은 `WHERE` 로 찾은 옛 트리플을 `DELETE` 로 빼면서 그 자리에 `INSERT` 로 새 트리플을 넣는 것입니다. 옛 값을 변수로 받아 `DELETE` 와 `WHERE` 에 같은 변수를 쓰면 **지금 무엇이 적혀 있든** 새 값으로 바뀝니다. `WHERE` 가 아무것도 못 찾으면 둘 다 일어나지 않아, "그 사실이 있을 때만 고친다"가 따라옵니다.

In [ ]:
# 빈 그래프를 하나 만들어 넣고 지우는 것만 확인한다(우리 그래프를 건드리지 않으려고)
scratch_graph = Graph()
scratch_graph.bind('ex', EX)
# INSERT DATA 는 적어 준 트리플을 그대로 넣는다(패턴이 아니라 값이라 DATA 다)
scratch_graph.update(QUERY_PREFIX + 'INSERT DATA { ex:모나리자 ex:제작연도 "1503" . }')
print('넣은 뒤:', len(scratch_graph))   # 출력: 1

In [ ]:
# DELETE DATA 는 같은 트리플을 지운다
scratch_graph.update(QUERY_PREFIX + 'DELETE DATA { ex:모나리자 ex:제작연도 "1503" . }')
print('지운 뒤:', len(scratch_graph))   # 출력: 0

> `"1503"` 처럼 따옴표로 적은 값이 3절에서 말한 **리터럴**입니다. 개체가 아니라 값이라 주소를 붙이지 않습니다.

`INSERT DATA`·`DELETE DATA` 는 **적어 준 트리플 그대로**를 넣고 뺍니다. 조건에 맞는 것을 한꺼번에 고치려면 `WHERE` 를 붙인 형태를 씁니다. "찾아서 고친다"가 되는 셈입니다.

In [ ]:
# 연습용 사본을 하나 만들어 거기서만 고친다(우리 graph 는 그대로 둔다)
practice_graph = Graph()
practice_graph.bind('ex', EX)
for fact in graph:
    practice_graph.add(fact)
# DELETE ... WHERE ... : 조건에 맞는 트리플을 찾아 한꺼번에 지운다
practice_graph.update(QUERY_PREFIX + """
DELETE { ?person ex:국적 ?country . }
WHERE  { ?person ex:국적 ?country . }
""")
# 출력: 국적 트리플이 통째로 빠져 원래보다 줄어든다
print('국적을 모두 지운 뒤:', len(practice_graph), '/ 원래', len(graph))

In [ ]:
# INSERT ... WHERE ... : 조건에 맞는 것을 찾아 새 트리플을 한꺼번에 넣는다
practice_graph.update(QUERY_PREFIX + """
INSERT { ?person ex:국적있음 "아니오" . }
WHERE  { ?person a ex:인물 . }
""")
# 출력: 인물 11명에게 표시가 하나씩 붙어 그만큼 늘어난다
print('표시를 붙인 뒤:', len(practice_graph))

### 🖐️ 함께 따라하기: 미술관 그래프에 SPARQL 보내기

> 이어서 **미술관 소장품 그래프**(`museum_graph`)로 같은 일을 합니다.

이제 `museum_graph` 에 SPARQL 을 보내 **작품마다 그 작가와 작가의 국적**을 찾아 출력하세요. 패턴 두 줄을 쓰고, 작가 자리에 **같은 변수**를 두어 이어야 합니다.

- 첫 패턴: 작품의 `창작자` 가 누구인지
- 둘째 패턴: **그 사람**의 `국적` 이 어디인지
- 결과는 작품 이름 기준으로 정렬해 `작품 -> 작가 -> 국가` 꼴로 출력

**확인 기준**: **5줄**이 나오고, 모나리자 줄에 `레오나르도 다 빈치` 와 `피렌체 공화국` 이 보이면 맞습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) SELECT 에 작품·작가·국가 세 변수를 적는다
# 2) WHERE 에 창작자 패턴과 국적 패턴을 두 줄로 적고, 작가 자리에 같은 변수를 쓴다
# 3) museum_graph.query 로 실행해 세 값을 이름으로 되돌려 출력한다

### 🖐️ 함께 따라하기: 두 조건 중 하나로 고르기

> 방금 배운 **`UNION`** 을 미술관 그래프에 씁니다.

`museum_graph` 에서 **국적이 네덜란드 왕국(`ex:네덜란드_왕국`)이거나 노르웨이(`ex:노르웨이`)인 작가**가 만든 작품을 찾아 `작품 -> 작가` 꼴로 출력하세요.

- 첫 패턴: 작품의 `창작자` 가 누구인지
- 그 사람의 국적을 두 갈래로 적고 `UNION` 으로 잇는다
- 작품 이름 기준으로 정렬

**확인 기준**: **2줄**이 나오고 `별이 빛나는 밤` 과 `절규` 가 보이면 맞습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) SELECT 에 작품·작가 두 변수를 적는다
# 2) WHERE 에 창작자 패턴을 적고, 국적 두 갈래를 중괄호로 감싸 UNION 으로 잇는다
# 3) museum_graph.query 로 실행해 두 값을 이름으로 되돌려 출력한다

### ✅ 바로 확인 퀴즈

**1.** 패턴 `?person ex:직업 ?job .` 은 어떤 트리플을 찾나요?

<details><summary>정답 보기</summary>

술어가 `직업` 인 **모든** 트리플입니다(주어·목적어는 아무거나). 즉 "누가 무슨 직업인가"를 전부 찾습니다.

</details>

**2.** 패턴 두 줄에 **같은 변수 이름**을 쓰면 무슨 일이 일어나나요?

<details><summary>정답 보기</summary>

그 변수 자리에는 **두 패턴을 모두 만족하는 값**만 남습니다. 첫 패턴이 찾아낸 값 중에서 둘째 패턴도 만족하는 것만 결과에 나오므로, 관계형 데이터베이스의 **조인**과 같은 일이 됩니다.

</details>

**3.** `OPTIONAL { ?person ex:국적 ?country . }` 을 `OPTIONAL` 없이 그냥 `?person ex:국적 ?country .` 로 바꾸면 결과가 어떻게 달라지나요?

<details><summary>정답 보기</summary>

국적이 적혀 있지 않은 사람은 **결과에서 통째로 사라집니다.** 패턴을 나란히 적는 것은 "둘 다 있어야 한다"는 뜻이라 국적 트리플이 없는 사람은 그 패턴에서 떨어지기 때문입니다. `OPTIONAL` 은 "있으면 채우고 없으면 빈칸"이라 행을 남깁니다.

</details>

**4.** `?work ex:창작자/ex:국적 ?country .` 한 줄은 패턴 두 줄로 쓰면 어떻게 되나요? 두 방식의 결과에는 어떤 차이가 있나요?

<details><summary>정답 보기</summary>

`?work ex:창작자 ?person .` 과 `?person ex:국적 ?country .` 두 줄이 됩니다. **답(작품과 국가의 짝)은 같습니다.** 다른 점은 중간에 거쳐 간 작가를 **변수로 붙잡아 두느냐**입니다. 경로로 쓰면 중간 값이 남지 않아 결과에 넣을 수 없고, 두 줄로 쓰면 `?person` 을 `SELECT` 에 넣어 함께 볼 수 있습니다.

</details>

---
# 5. 공개 지식 그래프에 직접 묻기

## 왜 필요할까요?
4절에서 쓴 그래프는 사실이 마흔 개도 안 됩니다. 진짜 힘은 **남이 만들어 둔 거대한 지식 그래프에 같은 언어로 물어볼 수 있다**는 데 있습니다. 우리가 맨 앞에서 데이터를 받아 온 것도 사실은 이 일을 한 것이었습니다. 이제 그 질의문을 직접 읽고 씁니다.

## Wikidata 를 읽는 법: Q-id 와 P-id
Wikidata 는 이름 대신 **번호**를 씁니다. 항목에는 `Q`, 속성에는 `P` 가 붙습니다.

| 종류 | 예 | 뜻 |
|---|---|---|
| 항목(Q-id) | `Q937` | 알베르트 아인슈타인 |
| 항목(Q-id) | `Q169470` | 물리학자 |
| 속성(P-id) | `P27` | 국적 |
| 속성(P-id) | `P106` | 직업 |

이 번호를 주소로 펼쳐 주는 **접두사**가 `wd:`·`wdt:` 입니다.

| 접두사 | 펼치면 | 가리키는 것 |
|---|---|---|
| `wd:` | `http://www.wikidata.org/entity/` | 항목(사람·사물). `wd:Q937` = 알베르트 아인슈타인 |
| `wdt:` | `http://www.wikidata.org/prop/direct/` | 속성(관계). `wdt:P27` = 국적 |

이름 대신 번호를 쓰는 이유는 **같은 이름을 가진 사람이 여럿이어도 번호는 하나뿐**이기 때문입니다. 우리가 맨 앞에서 본 `PEOPLE_QIDS` 가 바로 그 번호였습니다.

## 브라우저에서 직접 질의해 보기

코드로만 물어볼 수 있는 것이 아닙니다. 아래 두 곳은 **웹 페이지에 질의문을 붙여 넣고 실행 버튼을 누르면** 결과를 표로 보여 줍니다. 오타가 나면 그 자리에서 알려 주므로, 질의문을 다듬을 때는 여기서 먼저 맞춰 본 뒤 노트북으로 옮기는 편이 빠릅니다.

| 사이트 | 주소 | 특징 |
|---|---|---|
| Wikidata Query Service | <https://query.wikidata.org> | 위키미디어 공식. 결과를 표·지도·그래프로 바꿔 볼 수 있고, `SERVICE wikibase:label` 이 동작합니다. 예시 질의 모음(`Examples`)도 여기 있습니다 |
| QLever | <https://qlever.dev/wikidata> | 우리가 코드에서 쓰는 그 엔진의 웹 화면. 빠르지만 `SERVICE wikibase:label` 은 지원하지 않아 이름은 `rdfs:label` 로 직접 가져와야 합니다 |

> **번호를 모를 때**: `https://www.wikidata.org/wiki/Q937` 처럼 항목 번호를 주소에 넣으면 그 항목의 설명 페이지가 열립니다. 반대로 이름으로 찾고 싶으면 위키데이터 사이트에서 검색한 뒤 주소 끝의 `Q...` 를 읽으면 됩니다.

> 아래 셀들이 보내는 질의문은 그대로 복사해 두 사이트에 붙여 넣어도 같은 답이 나옵니다. **같은 질의문이 코드에서도 브라우저에서도 통한다**는 것이 표준 질의 언어의 값어치입니다.

## 질의문 한 줄씩 읽기

Wikidata 공식 엔드포인트에 보내는 질의문의 전형적인 모습입니다. 한 줄씩 뜯어 봅니다.

```sparql
SELECT ?x ?xLabel ?country ?countryLabel WHERE {
  # 1) 물어볼 대상을 세 사람으로 좁힌다
  VALUES ?x { wd:Q937 wd:Q935 wd:Q7186 }
  # 2) 패턴 1: 그 사람의 직업(P106)이 물리학자(Q169470)인가
  ?x wdt:P106 wd:Q169470 .
  # 3) 패턴 2: 같은 ?x 의 국적(P27)을 ?country 에 담는다
  ?x wdt:P27  ?country .
  # 4) 이름 자동 채우기(공식 엔드포인트 전용 기능)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ko,en". }
}
# 5) 사람 순, 같은 사람 안에서는 국가 순으로 정렬
ORDER BY ?x ?country
```

| 조각 | 읽는 법 |
|---|---|
| `VALUES ?x { ... }` | `?x` 에 넣어 볼 값을 **직접 나열**한다. 이걸 빼면 Wikidata 전체를 뒤지므로 결과가 엄청나게 커진다 |
| `?x wdt:P106 wd:Q169470 .` | "?x 의 직업은 물리학자" 라는 트리플 패턴. 마침표가 한 패턴의 끝 |
| `?x wdt:P27 ?country .` | 목적어 자리를 변수로 두면 **그 값을 받아 온다**(질문이 된다) |
| 두 줄의 같은 `?x` | 두 패턴을 잇는 **공통 변수**. 둘 다 만족하는 사람만 남는다(조인) |
| `?xLabel` | `SERVICE` 가 자동으로 만들어 주는 짝 변수. `?x` 의 이름이 여기 담긴다 |
| `SERVICE wikibase:label { ... "ko,en" }` | 항목마다 한국어 이름을, 없으면 영어 이름을 붙여 준다 |
| `ORDER BY ?x ?country` | 결과 정렬. 안 쓰면 순서가 실행할 때마다 달라질 수 있다 |

> **위 질의문에 `PREFIX` 가 없는 이유**: 공식 엔드포인트는 `wd:`·`wdt:`·`rdfs:` 같은 접두사를 **미리 선언해 두어** 생략할 수 있습니다. 반대로 QLever 나 rdflib 처럼 표준 SPARQL 만 아는 곳은 선언하지 않으면 질의를 **거부**합니다. 우리 코드가 매번 `PREFIX` 세 줄을 붙여 온 이유입니다.

> **`SERVICE wikibase:label` 은 표준 SPARQL 이 아닙니다.** Wikidata 공식 엔드포인트가 편의로 얹어 준 기능이라 다른 엔드포인트에서는 동작하지 않습니다. 아래에서 두 방식을 나란히 봅니다.

이제 이 질의문을 **공식 엔드포인트에 그대로 보내** 봅니다.

In [ ]:
# 위 질의문을 Wikidata 공식 엔드포인트로 보낸다
official_query = """
SELECT ?x ?xLabel ?country ?countryLabel WHERE {
  VALUES ?x { wd:Q937 wd:Q935 wd:Q7186 }
  ?x wdt:P106 wd:Q169470 .
  ?x wdt:P27  ?country .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ko,en". }
}
ORDER BY ?x ?country
"""
# 공식 엔드포인트는 요청이 몰리면 429(요청이 너무 잦음)로 거절한다.
# 거절당했을 때 무슨 일이 일어났는지 보려고 예외를 잡아 메시지를 찍는다
try:
    official_rows = run_sparql(official_query, endpoint=WIKIDATA)
    print('공식 엔드포인트에서 받은 행 수:', len(official_rows))
except Exception as error:
    official_rows = []
    print('공식 엔드포인트가 거절했습니다:', error)
    print('429 라면 요청 제한입니다. 잠시 뒤 다시 실행하거나 아래 표준 SPARQL 판으로 이어 가세요')

In [ ]:
# 받은 행을 읽는다. 거절당했으면 이 셀은 아무것도 찍지 않는다
for row in official_rows:
    # ?x 자리에 온 값은 이름이 아니라 주소(URI)다. 이름은 ?xLabel 에 따로 담겨 온다
    print(row['x']['value'], row['xLabel']['value'], '->', row['countryLabel']['value'])

> **거절 메시지가 찍혔더라도 실패가 아닙니다.** 공식 엔드포인트는 전 세계가 함께 쓰는 무료 서비스라 **한 주소에서 오는 요청 수를 제한**합니다. 한도를 넘으면 `HTTP Error 429` 로 거절합니다(강의장은 여러 사람이 같은 인터넷 주소를 쓰므로 특히 잘 걸립니다). 공개 API 를 쓸 때는 이렇게 **거절당하는 경우까지 코드가 감당**해야 하고, 대개 잠시 뒤 재시도하거나 다른 엔드포인트로 갈아탑니다. 우리는 아래에서 갈아탑니다.

같은 질문을 **표준 SPARQL 만으로** 써 보면 `SERVICE` 가 무엇을 대신해 주고 있었는지 분명해집니다. 이름은 `rdfs:label` 로 직접 가져오고, 여러 언어 중 한국어만 남기도록 `FILTER` 를 겁니다. 이 판은 어느 엔드포인트에서나 동작합니다.

In [ ]:
# 같은 질문의 표준 SPARQL 판. 이름을 rdfs:label 로 직접 가져온다
standard_query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?x ?name ?countryName WHERE {
  VALUES ?x { wd:Q937 wd:Q935 wd:Q7186 }
  ?x wdt:P106 wd:Q169470 .
  ?x wdt:P27  ?country .
  ?x       rdfs:label ?name .        FILTER(lang(?name) = "ko")
  ?country rdfs:label ?countryName . FILTER(lang(?countryName) = "ko")
}
ORDER BY ?name ?countryName
"""
for row in run_sparql(standard_query):
    # 출력 10줄 안팎: 마리 퀴리 3줄 + 알베르트 아인슈타인 7줄. 아이작 뉴턴은 한 줄도 없다
    # 앞 칸이 ?x 에 실제로 담긴 값(주소)이고, 뒤 두 칸이 사람이 읽으라고 붙인 이름이다
    print(row['x']['value'], row['name']['value'], '->', row['countryName']['value'])

> 세 가지가 눈에 띕니다.
>
> 1) `?x` 자리에 온 값이 **이름이 아니라 주소**입니다(`http://www.wikidata.org/entity/Q937`). 이름은 사람이 읽으라고 `?xLabel` 이나 `rdfs:label` 로 따로 붙여 준 것입니다.
> 2) **같은 사람이 여러 줄**에 나옵니다. 아인슈타인은 국적으로 적힌 항목이 여럿이기 때문입니다.
> 3) **아이작 뉴턴(`wd:Q935`)은 한 줄도 없습니다.** 원본에서 뉴턴의 직업에 물리학자가 적혀 있지 않아 첫 패턴에서 떨어집니다. 4절에서 우리 그래프를 조회했을 때와 **같은 결과**입니다.

마지막으로 원본이 얼마나 큰지 세어 봅시다. `COUNT` 로 개수만 받아 오면 데이터를 다 내려받지 않고도 규모를 알 수 있습니다.

In [ ]:
# 원본에 물리학자가 몇 명이나 등록돼 있는지 - COUNT 는 개수만 돌려주므로 응답이 한 줄이다
count_query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
SELECT (COUNT(DISTINCT ?person) AS ?count) WHERE {
  ?person wdt:P106 wd:Q169470 .
}
"""
count_rows = run_sparql(count_query)
# AS 로 붙인 이름(?count)이 그대로 결과의 키가 된다
print('Wikidata 에 등록된 물리학자 수:', count_rows[0]['count']['value'])   # 출력: 59000 안팎

> 우리 그래프는 물리학자가 네 명, 원본은 5만 명이 넘습니다. **같은 질의문**이 양쪽에서 그대로 동작한다는 점이 표준의 힘입니다.

## 계층을 타고 올라가기: `*` 경로

4절에서 미뤄 둔 `*` 를 여기서 씁니다. Wikidata 의 직업은 **계층**으로 정리돼 있습니다. "이론물리학자"·"천체물리학자"는 각각 "물리학자"의 **하위 개념**(`wdt:P279`, 상위 개념)이라고 적혀 있습니다. 먼저 그 하위 목록부터 봅니다.

In [ ]:
# 물리학자(Q169470)를 상위 개념으로 두는 직업들. P279 = 상위 개념
subclass_query = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?name WHERE {
  ?job wdt:P279 wd:Q169470 .
  ?job rdfs:label ?name . FILTER(lang(?name) = "ko")
}
ORDER BY ?name
"""
subclass_rows = run_sparql(subclass_query)
# 출력: 16개 안팎 - 이론물리학자·천체물리학자·핵물리학자 같은 세부 분야가 나온다
print(len(subclass_rows), [r['name']['value'] for r in subclass_rows])

여기서 문제가 생깁니다. 어떤 사람의 직업에 **"이론물리학자"만** 적혀 있으면, `?p wdt:P106 wd:Q169470` 으로는 **잡히지 않습니다.** 계층을 타고 올라가 "물리학자의 하위 개념이면 물리학자로 친다"고 하려면 `wdt:P279*` 를 붙입니다. `*` 는 "그 술어를 **0홉 이상** 따라가라"는 뜻이라, 자기 자신도 포함합니다.

In [ ]:
# 같은 질문을 경로 없이 / 경로를 붙여 각각 세어 본다
COUNT_PREFIX = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
"""
direct_query = COUNT_PREFIX + 'SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p wdt:P106 wd:Q169470 . }'
# wdt:P106/wdt:P279* = 직업으로 1홉 간 뒤, 상위 개념을 0홉 이상 타고 올라간다
path_query = COUNT_PREFIX + 'SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p wdt:P106/wdt:P279* wd:Q169470 . }'
print('직업이 물리학자 그대로:', run_sparql(direct_query)[0]['n']['value'])   # 출력: 59000 안팎
print('하위 직업까지 포함  :', run_sparql(path_query)[0]['n']['value'])   # 출력: 72000 안팎

> 만 명 넘게 늘었습니다. **온톨로지의 계층이 실제로 답을 바꾼다**는 것을 숫자로 보여 주는 장면입니다. 2절에서 우리가 직업 여덟 가지를 손으로 정한 것도 같은 문제를 다룬 것이었습니다. 다만 우리는 계층 대신 **목록**으로 범위를 정했습니다.

### 🖐️ 함께 따라하기: 조건을 바꿔 원본에 물어보기

> 이번에는 **우리 목록에 없는 사람들**을 원본에서 찾아봅니다.

`run_sparql` 로 **국적이 네덜란드 왕국(`wd:Q29999`)이고 직업이 화가(`wd:Q1028181`)인 사람**의 한국어 이름을 **10명만** 받아 출력하세요.

- `VALUES` 는 쓰지 않습니다(대상을 좁히지 않고 원본 전체에서 찾습니다)
- 패턴 두 줄에 **같은 변수**를 써서 두 조건을 모두 만족하는 사람만 남깁니다
- 이름은 `rdfs:label` 로 가져오고 `FILTER(lang(...) = "ko")` 로 한국어만 남깁니다
- 결과가 너무 많으므로 질의문 맨 끝에 `LIMIT 10` 을 붙입니다

**확인 기준**: 이름이 **10줄** 출력되면 맞습니다(누가 나오는지는 원본 상태에 따라 달라집니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) PREFIX 세 줄(wd·wdt·rdfs)로 질의문을 시작한다
# 2) WHERE 에 직업 패턴·국적 패턴을 같은 변수로 두 줄 적는다
# 3) rdfs:label 로 이름을 가져오고 FILTER 로 한국어만 남긴다
# 4) ORDER BY 로 정렬하고 LIMIT 10 을 붙인 뒤 run_sparql 로 실행해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `wd:Q937` 과 `wdt:P27` 은 각각 무엇을 가리키나요?

<details><summary>정답 보기</summary>

`wd:Q937` 은 **항목**(알베르트 아인슈타인), `wdt:P27` 은 **속성**(국적)입니다. Wikidata 는 항목에 `Q`, 속성에 `P` 번호를 붙이고, `wd:`·`wdt:` 접두사가 그 번호를 실제 주소로 펼쳐 줍니다.

</details>

**2.** 같은 질의문인데 `SERVICE wikibase:label` 을 쓴 판과 `rdfs:label` 을 쓴 판이 따로 있는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

`SERVICE wikibase:label` 은 **Wikidata 공식 엔드포인트가 편의로 얹어 준 기능**이라 다른 엔드포인트에서는 동작하지 않습니다. 어디서나 돌아가게 하려면 `rdfs:label` 로 이름을 직접 가져오는 **표준 SPARQL** 로 써야 합니다.

</details>

---
## 🚀 응용 클론코딩: 직업별 인원 세기

4절에서 만든 `graph` 에 SPARQL 을 보내 **직업마다 사람이 몇 명인지** 세어 출력하세요.

- `SELECT` 에 직업 변수와 개수를 담습니다. 개수는 `(COUNT(?변수) AS ?count)` 꼴로 적습니다
- `GROUP BY` 로 직업마다 묶습니다
- 인원이 많은 순으로 정렬합니다(`ORDER BY DESC(?count)`)
- 직업 이름은 URI 이므로 마지막 칸만 잘라 출력합니다

**확인 기준**: 여덟 직업이 모두 나오고 **물리학자가 4명으로 가장 많으면** 맞습니다.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) SELECT 에 ?job 과 (COUNT(?person) AS ?count) 를 적는다
# 2) WHERE 에 직업 패턴 한 줄을 적고 GROUP BY ?job 으로 묶는다
# 3) ORDER BY DESC(?count) 로 정렬한 뒤 graph.query 로 실행해 출력한다

---
## 이번 강의 정리

| 개념 | 핵심 |
|---|---|
| 지식 그래프 | 사실(연결)을 모아 쌓은 그래프. "이름"이 아니라 "연결"이 지식 |
| 온톨로지 | 담을 범위를 정한 규격. **관계 규칙**(어떤 타입끼리 잇는가)과 **타입 사전**(그 타입에 어떤 값이 속하는가)을 둘 다 적어야 선다 |
| RDF 트리플 | 주어-술어-목적어 표준 형식. LPG 와 같은 그래프의 다른 표기 |
| SPARQL | RDF 에 묻는 표준 쿼리 언어. `SELECT ?x WHERE { 패턴 }` |
| 조인 | 패턴 여러 줄에 **같은 변수**를 쓰면 두 조건을 다 만족하는 것만 남는다 |
| 조건·다듬기 | `FILTER`·`UNION`(또는)·`OPTIONAL`(없으면 빈칸)·`DISTINCT`·`ORDER BY`·`LIMIT` |
| 속성 경로 | `술어1/술어2`(2홉을 한 줄로)·`술어*`(계층을 0홉 이상 타고 올라가기) |
| 질의 종류 | `SELECT`(표)·`ASK`(참·거짓)·`CONSTRUCT`(새 그래프)·`INSERT`·`DELETE`(넣고 빼기)·`DELETE`+`INSERT`+`WHERE`(수정. `UPDATE` 문장은 따로 없다) |
| RDF·RDFS | `a`(=`rdf:type`)는 **개체**가 어느 타입인지를, `rdfs:` 는 **타입 쪽**을 적는다. `rdfs:Class`(타입 선언)·`rdfs:subClassOf`(상하 관계)·`rdfs:domain`·`rdfs:range`(술어가 잇는 타입)·`rdfs:label`(이름) |
| URI 식별 | 진짜 RDF 는 이름이 아니라 주소로 가리킨다(`wd:Q937`). 동명이인이 섞이지 않는다 |
| 엔드포인트 | 질의를 받아 주는 웹 주소. 같은 데이터라도 기능과 요청 제한이 다르다 |

- 지식 그래프는 **사실 = (주어, 술어, 목적어)** 를 이어 붙인 것입니다.
- 원본은 우리 생각보다 훨씬 잘고 지저분합니다. 온톨로지는 **담을 범위를 정하는 규격**입니다.
- 같은 SPARQL 질의문이 **내 그래프(rdflib)** 에서도, **공개 엔드포인트**에서도 그대로 돕니다.

## 다음 시간 준비: Neo4j 받아 두기

다음 시간에는 그래프 전용 데이터베이스 **Neo4j** 를 설치해서 씁니다. 수업 시간에 내려받느라 기다리지 않도록 **미리 받아 두세요**(설치·인스턴스 만들기·연결은 다음 시간에 함께 합니다).

- **권장: Neo4j Desktop(내 컴퓨터에 설치)**: <https://neo4j.com/deployment-center/> 에서 내려받습니다(Windows·macOS·Linux). 내려받기 화면에 표시되는 **Activation Key** 를 복사해 두세요. 첫 실행 때 한 번 붙여 넣습니다.
- **설치가 어렵다면: Neo4j Aura(클라우드, 설치 없이)**: <https://neo4j.com/product/auradb/> 에서 무료 인스턴스를 만들 수 있습니다. 계정이 필요하고, 만들 때 뜨는 **비밀번호는 한 번만 표시**되니 반드시 저장하세요.
- 어느 쪽이든 **직접 정한 비밀번호를 기억해 두세요**. 다음 시간에 파이썬에서 접속할 때 씁니다.
- 회사·학교 노트북이라 설치가 막혀 있다면 미리 알려 주세요(Aura 로 안내합니다).

## ⏭️ 예고: 다음 단원

오늘은 남이 운영하는 지식 그래프에 **질의**만 했습니다. 다음 단원부터는 그래프 전용 데이터베이스인 **Neo4j** 를 설치해, 수백만 개의 노드·관계를 **직접 저장하고** 쿼리로 찾는 진짜 그래프DB 를 다룹니다.

수고하셨습니다!